<a href="https://colab.research.google.com/github/fred-creator-creat/bradesco-assistente-voz-whisper/blob/main/Desafio_Bradesco_Assistente_GenAI_Whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalando as bibliotecas necessárias
!pip install git+https://github.com/openai/whisper.git -q
!pip install gTTS -q
!pip install -q -U google-generativeai

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [2]:
!pip install --upgrade click

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.


In [3]:
import whisper
from gtts import gTTS
import google.generativeai as genai
print("Tudo certo! Pode prosseguir.")

Tudo certo! Pode prosseguir.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
import google.generativeai as genai
from google.colab import userdata # <--- Linha para ler o Secret
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
import os

# --- CONFIGURAÇÃO DA CHAVE SEGURA ---
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)
model_gemini = genai.GenerativeModel('gemini-2.5-flash')

# --- FUNÇÃO PARA GRAVAR ÁUDIO ---
RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = b64decode(js_result.split(',')[1])
    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)
    return f'/content/{file_name}'

print("Célula 4 configurada com SUCESSO e SEGURANÇA!")

Célula 4 configurada com SUCESSO e SEGURANÇA!


In [6]:
import whisper
from gtts import gTTS
import datetime
import time

# Carregando o modelo Whisper
print("Carregando o ouvido (Whisper)... aguarde.")
model_whisper = whisper.load_model("base")

print("\n--- Assistente Pronta! ---")
print("Diga 'Sair', 'Encerrar' ou 'Tchau' para parar.")

while True:
    try:
        print("\nOuvindo... (Fale agora)")
        audio_path = record(sec=5)

        print("Processando sua fala...")
        result = model_whisper.transcribe(audio_path, fp16=False, language='pt')
        texto_usuario = result["text"].strip().lower() # Forçamos tudo para minúsculo
        print(f"Você disse: {texto_usuario}")

        # Lógica de ENCERRAMENTO mais forte (limpando pontos e espaços)
        texto_limpo = texto_usuario.replace(".", "").replace("!", "").strip()
        palavras_parar = ["sair", "encerrar", "parar", "tchau", "desligar", "finish"]

        if any(p in texto_limpo for p in palavras_parar):
            print("Entendido! Encerrando a assistente. Até a próxima!")
            despedida = gTTS(text="Até a próxima, Fred! Encerrando.", lang='pt')
            despedida.save("tchau.wav")
            display(Audio("tchau.wav", autoplay=True))
            break

        if not texto_limpo:
            print("Não ouvi nada. Tente falar novamente.")
            continue

        # CORREÇÃO DO HORÁRIO (Ajustando para Brasília -3 horas)
        # Pegamos a hora do servidor e tiramos 3 horas
        fuso_horario = datetime.timezone(datetime.timedelta(hours=-3))
        hora_brasilia = datetime.datetime.now(fuso_horario).strftime("%H:%M")

        instrucao = f"Você é uma assistente virtual. O horário atual de Brasília é {hora_brasilia}. Responda de forma curta: "

        response = model_gemini.generate_content(instrucao + texto_usuario)
        resposta_final = response.text
        print(f"Assistente: {resposta_final}")

        # Voz
        voz = gTTS(text=resposta_final, lang='pt', slow=False)
        voz.save("resposta.wav")
        display(Audio("resposta.wav", autoplay=True))

        time.sleep(4) # Espera um pouco mais para o áudio tocar

    except Exception as e:
        print(f"Ocorreu um erro: {e}")
        break

Carregando o ouvido (Whisper)... aguarde.

--- Assistente Pronta! ---
Diga 'Sair', 'Encerrar' ou 'Tchau' para parar.

Ouvindo... (Fale agora)


<IPython.core.display.Javascript object>

Processando sua fala...
Você disse: que horas são?
Assistente: São 08:15.



Ouvindo... (Fale agora)


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Processando sua fala...
Você disse: hoje...
Assistente: hoje é de manhã.



Ouvindo... (Fale agora)


<IPython.core.display.Javascript object>

Processando sua fala...
Você disse: tchau!
Entendido! Encerrando a assistente. Até a próxima!
